![bookstore](bookstore.jpg)


Identifying popular products is incredibly important for e-commerce companies! Popular products generate more revenue and, therefore, play a key role in stock control.

You've been asked to support an online bookstore by building a model to predict whether a book will be popular or not. They've supplied you with an extensive dataset containing information about all books they've sold, including:

* `price`
* `popularity` (target variable)
* `review/summary`
* `review/text`
* `review/helpfulness`
* `authors`
* `categories`

You'll need to build a model that predicts whether a book will be rated as popular or not.

They have high expectations of you, so have set a target of at least 70% accuracy! You are free to use as many features as you like, and will need to engineer new features to achieve this level of performance.

In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

speech_df = pd.read_csv("data/books.csv")

In [54]:
speech_df["review/text"] = speech_df["review/text"].str.lower()
speech_df["char_cnt"]     = speech_df["review/text"].str.len()
speech_df["word_cnt"]     = speech_df["review/text"].str.split().str.len()
speech_df["avg_word_len"] = speech_df["char_cnt"] / speech_df["word_cnt"]

In [55]:
X = speech_df.drop(columns=["popularity"])
y = speech_df["popularity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [56]:
tv = TfidfVectorizer(max_features=100, stop_words='english')

tv = TfidfVectorizer(max_features=5000, stop_words='english')
tv.fit(X_train["review/text"])

train_tv_df = pd.DataFrame(
    tv.transform(X_train["review/text"]).toarray(),
    columns=tv.get_feature_names_out()
)

print("Top 15 TF-IDF tokens in the first training row:")
print(train_tv_df.iloc[0].sort_values(ascending=False).head(15))

Top 15 TF-IDF tokens in the first training row:
coltrane        0.733638
jazz            0.384698
political       0.196837
black           0.135972
john            0.134976
music           0.107804
1960s           0.100492
questionable    0.095449
ideas           0.090783
critics         0.088369
interviews      0.084463
influence       0.080517
movement        0.080034
scene           0.072764
analysis        0.071936
Name: 0, dtype: float64


In [57]:
text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=2000, stop_words='english'))
])

num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('text', text_transformer, 'review/text'),
    ('num',  num_transformer,  ["char_cnt", "word_cnt", "avg_word_len"])
])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000))
])
param_grid = {
    'preprocessor__text__tfidf__max_features': [2000, 3000, 5000],
    'classifier__C': [0.1, 1, 10]
}

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 9 candidates, totalling 45 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('text',
                                                                         Pipeline(steps=[('tfidf',
                                                                                          TfidfVectorizer(max_features=2000,
                                                                                                          stop_words='english'))]),
                                                                         'review/text'),
                                                                        ('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['char_cnt',
                                                                          'word_cnt',
                                                                          'avg_word_len'])])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             n_jobs=-1,
             param_grid={'classifier__C': [0.1, 1, 10],
                         'preprocessor__text__tfidf__max_features': [2000, 3000,
                                                                     5000]},
             scoring='accuracy', verbose=1)

In [58]:
model_accuracy = grid_search.best_score_
print(model_accuracy)

0.7564214711729622
